In [1]:
import os
import re
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_curve, confusion_matrix
from sklearn.utils import resample
import joblib

In [2]:
# ---------- Feature Extraction ----------
def extract_features(text):
    words = text.split()
    num_words = len(words)
    keywords = ["lieferschein", "bestellnr", "lieferdatum", "kundennr", "iban", "mwst"]
    num_keywords_matched = sum(1 for kw in keywords if re.search(rf"\b{kw}\b", text, re.IGNORECASE))
    contains_email = int("e-mail" in text.lower())
    contains_kundennr = int("kundennr" in text.lower())
    num_dates = len(re.findall(r'\d{2}\.\d{2}\.\d{4}', text))

    return {
            'is_agb': int(num_words > 500),
            'contains_kundennr': contains_kundennr,
            'num_dates': num_dates,
            'num_keywords_matched': num_keywords_matched,
            'contains_email': contains_email
        }


# ---------- Process all PDFs ----------
def process_pdf_dir(folder_path="data/superbatch"):
    records = []
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(".pdf"):
            continue
        filepath = os.path.join(folder_path, filename)
        try:
            reader = PdfReader(filepath)
            for i, page in enumerate(reader.pages):
                text = page.extract_text() or ""
                features = extract_features(text)
                features['label'] = 1 if i == 0 else 0
                records.append(features)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
    return pd.DataFrame(records)

# ---------- Load and balance ----------
df = process_pdf_dir("data/superbatch")
df_0 = df[df['label'] == 0]
df_1 = df[df['label'] == 1]
min_size = min(len(df_0), len(df_1))
df_balanced = pd.concat([
    df_0.sample(min_size, random_state=42),
    df_1.sample(min_size, random_state=42)
]).sample(frac=1, random_state=42)

X = df_balanced.drop(columns=['label'])
y = df_balanced['label']

df

,is_agb,contains_kundennr,num_dates,num_keywords_matched,contains_email,label
0,0,0,0,0,1,1
1,0,0,1,1,1,1
2,0,0,2,0,1,1
3,0,0,1,1,0,1
4,0,0,1,1,0,0
...,...,...,...,...,...,...
96,0,0,1,0,0,0
97,0,0,1,1,0,1
98,0,0,3,0,1,1
99,0,0,3,1,0,1


In [3]:

def predict_with_threshold(model, X, threshold):
    proba = model.predict_proba(X)[:, 1]
    return (proba > threshold).astype(int)

# Balance the dataset
df_0 = df[df['label'] == 0]
df_1 = df[df['label'] == 1]

if len(df_0) < len(df_1):
    df_0_upsampled = resample(df_0, replace=True, n_samples=len(df_1), random_state=42)
    df_balanced = pd.concat([df_1, df_0_upsampled])
else:
    df_1_upsampled = resample(df_1, replace=True, n_samples=len(df_0), random_state=42)
    df_balanced = pd.concat([df_0, df_1_upsampled])

df_balanced = df_balanced.sample(frac=1, random_state=42)

# Split
X = df_balanced.drop(columns=['label'])
y = df_balanced['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Feature scaling with column preservation
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# Hyperparameter tuning
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs']}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1')
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_

# Threshold optimization
probas = best_model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, probas)
optimal_idx = (tpr - fpr).argmax()
optimal_threshold = thresholds[optimal_idx]

# Prediction
y_pred = predict_with_threshold(best_model, X_test_scaled, optimal_threshold)

# Evaluation
print("=== Tuned Logistic Regression ===")
print("Optimal Threshold:", round(optimal_threshold, 4))
print(classification_report(y_test, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Save model and scaler
joblib.dump(best_model, "logistic_model.pkl")
joblib.dump(scaler, "scaler.pkl")


=== Tuned Logistic Regression ===
Optimal Threshold: 0.5581
              precision    recall  f1-score   support

           0     0.7692    0.7692    0.7692        26
           1     0.7600    0.7600    0.7600        25

    accuracy                         0.7647        51
   macro avg     0.7646    0.7646    0.7646        51
weighted avg     0.7647    0.7647    0.7647        51

Confusion Matrix:
 [[20  6]
 [ 6 19]]


['scaler.pkl']